在深度学习的发展史上，LeNet（又称LeNet-5）是一个里程碑式的模型。

它由深度学习先驱Yann LeCun等人于1998年提出，被公认为第一个成功应用的卷积神经网络。尽管在LeNet之前已有类似卷积结构的研究，但LeNet首次将卷积、池化和反向传播结合，为现代计算机视觉奠定了基础。

Input: 1×32×32 图像（如手写数字）

C1: 卷积层，6个5×5卷积核 → 输出6×28×28

S2: 平均池化层 → 输出6×14×14

C3: 卷积层，16个5×5卷积核 → 输出16×10×10

S4: 平均池化层 → 输出16×5×5

C5: 卷积层，120个5×5卷积核 → 输出120×1×1（相当于全连接）

F6: 全连接层，输出84个神经元

Output: 全连接层，输出10类（数字0-9的概率）

注意： 输入为32×32图像（MNIST是28×28，用zero-padding补足） 使用平均池化（Average Pooling）而非现代常见的最大池化（Max Pooling） 激活函数为Sigmoid或tanh，而非ReLU

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class LeNet(nn.Module):
    def __init__(self):
        super(LeNet, self).__init__()
        # C1: 输入1通道，输出6通道，卷积核5x5
        self.conv1 = nn.Conv2d(1, 6, kernel_size=5)
        # S2: 平均池化层
        self.pool1 = nn.AvgPool2d(kernel_size=2, stride=2)
        # C3: 输入6通道，输出16通道
        self.conv2 = nn.Conv2d(6, 16, kernel_size=5)
        # S4: 平均池化
        self.pool2 = nn.AvgPool2d(kernel_size=2, stride=2)
        # C5: 全连接等价层（输入16×5×5 -> 输出120）
        self.conv3 = nn.Conv2d(16, 120, kernel_size=5)
        # F6: 全连接层
        self.fc1 = nn.Linear(120, 84)
        # Output: 输出10类
        self.fc2 = nn.Linear(84, 10)

    def forward(self, x):
        x = F.tanh(self.conv1(x))     # C1 + 激活
        x = self.pool1(x)             # S2
        x = F.tanh(self.conv2(x))     # C3 + 激活
        x = self.pool2(x)             # S4
        x = F.tanh(self.conv3(x))     # C5 + 激活
        x = x.view(-1, 120)           # 展平
        x = F.tanh(self.fc1(x))       # F6
        x = self.fc2(x)               # 输出层
        return x

LeNet 最初是为美国邮政局开发，用于识别手写邮政编码，是卷积神经网络（CNN）首次成功应用于现实世界问题。它在标准的 MNIST 数据集上取得了超过 99.2% 的准确率，远超当时的传统算法。与依赖人工特征设计的方法不同，LeNet 能通过卷积核自动学习图像特征，省去了手动构造特征的繁琐过程。它结构简单、效率高、准确率高，堪称 CNN 发展史上的里程碑。LeNet-5 首次系统性地引入了 Padding、卷积、池化和全连接等关键结构，这些核心思想至今仍是现代深度学习网络设计的基础。

11.2

2012年，深度学习迎来了历史性的一刻：AlexNet在ImageNet图像识别比赛（ILSVRC）中大放异彩，以压倒性优势击败传统计算机视觉方法，开启了深度学习在视觉领域的黄金时代。

2009 年，斯坦福大学的李飞飞教授（Fei-Fei Li）及其团队提出了 ImageNet 项目，旨在构建一个大规模、标注详尽的图像数据库。当时，计算机视觉领域的研究大多依赖于小规模、手工收集的数据集（如 CIFAR-10、MINIST 等），图像种类有限，标注数量稀缺，严重制约了深度学习模型的训练与泛化能力。ImageNet 最初收集了约 1,400 万张图像，并在其中挑选了 1,000 类常见物体进行精细标注，每类约 1,000 张图像。

2010 年起，ImageNet 大规模视觉识别挑战赛（ILSVRC，ImageNet Large Scale Visual Recognition Challenge）每年举办，成为深度学习领域的重要风向标。受到学者和企业的广泛关注。

在2012年之前，图像识别主要依赖手工特征和浅层模型，如SIFT特征、HOG描述符、支持向量机（SVM）等。然而，ImageNet比赛的复杂性（1000个类别、百万级图像）对传统方法提出了巨大挑战。

卷积神经网络虽然早已提出，但由于计算能力和数据量的限制，长期未能发挥优势。随着GPU的普及和大规模数据集的出现，训练深层卷积神经网络成为可能。AlexNet是由Alex Krizhevsky、Ilya Sutskever和Geoffrey Hinton在2012年提出的卷积神经网络架构，专为图像分类任务设计。它在ImageNet 2012挑战中将Top-5错误率从26%降低到了15%，领先第二名10个百分点，震惊业界。

In [ ]:
import torch
import torch.nn as nn

class AlexNet(nn.Module):
    def __init__(self, num_classes=1000):
        super(AlexNet, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 96, kernel_size=11, stride=4, padding=2),  # 输出: 96x55x55
            nn.ReLU(inplace=True),
            nn.LocalResponseNorm(size=5, alpha=1e-4, beta=0.75, k=2.0),
            nn.MaxPool2d(kernel_size=3, stride=2),  # 输出: 96x27x27

            nn.Conv2d(96, 256, kernel_size=5, padding=2),  # 输出: 256x27x27
            nn.ReLU(inplace=True),
            nn.LocalResponseNorm(size=5, alpha=1e-4, beta=0.75, k=2.0),
            nn.MaxPool2d(kernel_size=3, stride=2),  # 输出: 256x13x13

            nn.Conv2d(256, 384, kernel_size=3, padding=1),  # 输出: 384x13x13
            nn.ReLU(inplace=True),

            nn.Conv2d(384, 384, kernel_size=3, padding=1),  # 输出: 384x13x13
            nn.ReLU(inplace=True),

            nn.Conv2d(384, 256, kernel_size=3, padding=1),  # 输出: 256x13x13
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2)  # 输出: 256x6x6
        )
        self.classifier = nn.Sequential(
            nn.Dropout(p=0.5),
            nn.Linear(256 * 6 * 6, 4096),
            nn.ReLU(inplace=True),

            nn.Dropout(p=0.5),
            nn.Linear(4096, 4096),
            nn.ReLU(inplace=True),

            nn.Linear(4096, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), 256 * 6 * 6)
        x = self.classifier(x)
        return x

AlexNet的关键创新点

使用ReLU激活函数： 在当时，Sigmoid或Tanh是主流激活函数。AlexNet大胆采用了ReLU。相比Sigmoid，ReLU训练更快，不易梯度消失。

使用Dropout防止过拟合： 在全连接层引入Dropout技术，训练时随机丢弃部分神经元，大大减少了过拟合风险。

GPU并行训练： AlexNet使用两个GPU训练网络的不同部分，这在当时是很创新的做法，大大提升了训练效率。

局部响应归一化（LRN）： 在前两层使用了LRN来增强ReLU的响应效果，尽管这一技术后来被Batch Normalization取代，但在当时有效提升了模型性能。

应用了图像增强技术： AlexNet进行了数据增强，随机从256×256的原始图像中截取224×224大小区域，并以50%的概率做水平翻转，并且进行了光照扰动图像增强。

AlexNet的成功展示了“更深的网络 + 大量数据 + 强大计算力”三位一体的重要性。它的成功让深度学习从学术走向工业，促使Google、Facebook等巨头全面转向神经网络。它在ImageNet上的优异表现开启了CNN黄金时代，其后更多深层次卷积神经网络纷纷涌现，形成CNN快速迭代浪潮。并且它催生了硬件发展，让GPU成为训练深度模型的标配。

11.3

2014 年，Google 的研究团队提出了 GoogLeNet（又称 Inception V1），并凭借其在 ILSVRC 2014 中创纪录的 6.7% Top-5 错误率一举夺魁，几乎将前一年冠军的错误率腰斩。这一成果的关键在于 Inception 模块，它通过多尺度卷积和降维技巧，既增加了网络深度，又在计算量可控的前提下抑制了参数爆炸和过拟合。为了向之前的LeNet致敬，他们将Google里的L大写，让GoogLeNet里包含了一个LeNet。

在GoogLeNet中最重要的就是Inception模块，堆叠的多个Inception模块构成网络的主要框架。

输入是一个28×28×192的特征图，在设计卷积神经网络时，我们面临选择，接下来是进行池化操作还是卷积操作呢？对于卷积操作，我们应该选择多大的卷积核呢？1×1、3×3还是5×5呢？

Inception模块的思想，不做选择，全都要。接下来我们看具体操作。

1×1卷积

设置64个1×1×192的卷积核，输出特征图为28×28×64。

3×3卷积

设置128个3×3×192的卷积核，padding为1，stride为1，输出特征图为28×28×128。

5×5卷积

设置32个5×5×192的卷积核，padding为2，stride为1，输出特征图为28×28×32。

最大池化

设置最大池化窗口为2×2，padding为1，stride为1，（为了保持输出特征图的长宽不变，所以padding为1，stride为1）。输出特征图为28×28×192，然后再接32个1×1×192的卷积，改变输出特征图为28×28×32

上边的1×1、3×3、5×5、最大池化操作，输出的特征图的长和宽都是一致的，只是通道数不同，可以将它们按通道合并，最终的特征图尺寸为28×28×256。

Inception的思想是不做选择，每个模块做各种粒度的卷积，池化。但是这样带来的问题是参数量和计算量都非常大。我们以其中的5×5卷积核为例来看一下它的参数量和计算量。

32个5×5×192的卷积核的参数量为5×5×192×32=153600。接下来我们来看计算量，对于输出特征图里的每一个特征，都需要对一个卷积核5×5×192=4800个位置的特征和权重的乘法。累加我们不计入。然后输出特征图一共有28×28×32=25088个特征，所以总共计算量为4800×25088=120,422,400。

解决参数量和计算量大的问题，可以用1×1卷积来缩减维度，Inception模块在输入特征图和5×5卷积之间增加了16个1×1卷积核。这样它的总共参数量就为16×192+5×5×16×32=15872。参数量缩减为原来的十分之一。接着来看计算量：192×28×28×16+5×5×16×28×28×32=12,443,648和之前的计算量比，也是原来的十分之一。由此可见通过1×1卷积是可以大幅缩小模型的参数量和计算量的。

所以，Inception模块同样将1×1卷积应用在3×3卷积之前来降低输入的通道数。

通过上图可以看到GoogLeNet首先通过卷积，池化来提取原始图片的特征，然后是多个Inception模块，接一个MaxPolling来降低特征图的长和宽，然后再Inception模块，最终通过全连接层进行分类。

还有一个细节需要注意，就是上图中用红框标记的部分，在训练时也对中间隐藏层提取的特征直接进行分类训练，这样保证中间隐藏层提取的特征必须是对分类有帮助的。来增强模型训练的监督信号。

11.3.3 GoogLeNet的意义

多尺度并行卷积（Inception模块）：

Inception模块通过并行使用不同尺寸的卷积核（1x1、3x3、5x5）和池化操作，同时捕捉不同尺度的特征。这种设计突破了传统堆叠单一卷积层的模式，显著提升了模型的特征表达能力。

1x1卷积的革新应用：

利用1x1卷积进行通道降维和非线性增强（1×1卷积后的激活引入非线性）。这一技术成为后续模型设计的标配。

11.4

在 VGG Net 之前，AlexNet 等模型已经展示了深度网络在大规模图像分类任务中的潜力。然而，这些网络在卷积核大小、池化层和激活函数的设计上缺乏统一策略。VGGNet提出：通过使用多个连续的小卷积核（3×3）替代大卷积核（如 5×5 或 7×7），可以在保证感受野不变的前提下，提高网络的深度和表达能力，同时减少参数量。

3×3卷积核的优势

两个连续的3×3的卷积核产生的特征图上的每个特征的感受野是5×5，它和单独使用一个5×5卷积核是一样的。但是两个连续的3×3卷积核额外有两个好处：

参数量更少，假如输入通道数为
C
i
n
C 
in
​
 ,输出通道数为
C
o
u
t
C 
out
​
 ，则两个3×3卷积核的参数量为
2
×
3
×
3
×
C
i
n
×
C
o
u
t
2×3×3×C 
in
​
 ×C 
out
​
 ，等于
18
C
i
n
C
o
u
t
18C 
in
​
 C 
out
​
 。一个5×5卷积核的参数量为
5
×
5
×
C
i
n
×
C
o
u
t
5×5×C 
in
​
 ×C 
out
​
 ，等于
25
C
i
n
C
o
u
t
25C 
in
​
 C 
out
​
 。

网络层数更深，更容易提取高级特征。之前我们说过深度神经网络通过多层网络，可以逐层在之前层提取的特征基础上组合出更高级的特征，来提升网络表现，这一点也适用于卷积神经网络。

VGG Net 的核心思想是：

统一使用 3×3 卷积核：叠加两个 3×3 卷积层等价于一个 5×5 卷积层；叠加三个 3×3 等价于一个 7×7，且非线性增强。

网络深度可变：提出了 VGG-11、VGG-13、VGG-16、VGG-19 四种不同深度的变体，数字代表卷积或全连接层数量之和。

卷积层通过Padding来实现输入和输出特征图大小一致，只通过最大池化层来减少特征图尺寸。

统一的池化设计：在每个卷积块之后使用 2×2 最大池化层，下采样一半分辨率。

三层全连接：在卷积特征提取后，使用三层全连接，最后通过 Softmax 输出分类概率。

11.5

ResNet（残差神经网络）由何恺明等人于2015年提出，是深度学习领域里程碑式的工作。ResNet在2015年ILSVRC中以3.57%的top-5错误率夺冠，首次显著超越人类水平（5.1%）。其核心创新是残差学习，使得网络可以训练到极深的层次。在ResNet出现之前，一般的卷积神经网络只能达到30层，而有了ResNet之后，网络深度可以达到100多层，甚至1000层。

网络越深越好吗

之前我们讲过，同样的参数量，网络越深性能越好。因为神经网络可以逐层抽象特征，获取更好的拟合效果。

但是，在实际实验中，人们却发现，当网络到达一定深度后，随着网络深度的增加，训练误差反而上升。注意，这不是过拟合。过拟合是训练误差小，测试误差高。这里是训练误差会上升。这是因为深层网络难以优化导致的。

在ResNet的论文里作者也做了实验，对比一个20层的网络和56层的网络。他发现随着训练迭代（横轴），训练集上的错误率（下左图），测试集上的错误率（右下图）。20层的网络反而都优于56层的网络。

